<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/Titans_jax_Phase2_LM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Gemma-Titans Phase 2: LM Fine-Tuning

Фаза 2 обучения: переключаемся с послойной дистилляции на language modeling loss.

**Отличия от Phase 1:**
- Выходы студента передаются между слоями (нет `stop_gradient`, нет teacher chain)
- Loss: cross-entropy по токенам вместо per-layer MSE
- `training_phase=2` в конфиге модели
- Веса загружаются из `saved_titans_delta` (результат Phase 1)
- Сниженные LR (веса уже предобучены)

In [ ]:
# 0. Environment Setup
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog
!pip install -q flax optax seqio
!pip install importlib_resources

In [ ]:
!git clone https://github.com/andrew-veriga/Titans_jax.git

## Start

In [ ]:
import os
os._exit(0)

In [ ]:
import sys
import os
sys.path.append(os.getcwd())

import jax
import jax.numpy as jnp
import optax
import dataclasses
import numpy as np
import os
import orbax.checkpoint as ocp
import shutil

from kauldron import kd
from kauldron.ktyping import Array
from gemma import gm
from gemma.gm.nn import _config

# Our custom Titans integration
import importlib

%cd /content/Titans_jax
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
""" старые настройки
# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"
"""
# Разрешаем JAX забрать память сразу для максимальной скорости
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"

# Увеличиваем долю памяти (оставляем чуть-чуть на системные нужды)
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"

# Оптимизируем флаги для производительности, а не для экономии
os.environ["XLA_FLAGS"] = (
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_joint_all_gather_opt=true "
    "--xla_tpu_enable_latency_hiding_scheduler=true " # Скрывает задержки памяти за вычислениями
    "--xla_tpu_all_reduce_combine_threshold_bytes=134217728" # Оптимально для больших батчей
)

os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

In [ ]:
import psutil
import os

# Get system memory stats
ram_info = psutil.virtual_memory()
total_ram_gb = ram_info.total / (1024**3)
available_ram_gb = ram_info.available / (1024**3)

print(f"Total RAM: {total_ram_gb:.2f} GB")
print(f"Available RAM: {available_ram_gb:.2f} GB")

# Check if High-RAM is likely active
if total_ram_gb > 170:
    print("\nStatus: Likely High-RAM runtime")
else:
    print("\nStatus: Standard RAM runtime")



## 1. Загрузка весов из Phase 1

In [ ]:
!cp "/content/drive/Shareddrives/shared veriga/saved_titans_delta/saved_titans_delta.zip" saved_titans_delta.zip

## 2. Гиперпараметры

In [ ]:
batch_size = 4
max_length = 2048
total_steps = 20000

titans_phase2_first_layer = 11  # Titans layers from this index onward are active.
                                 # Earlier layers revert to standard Gemma blocks.
                                 # 17 → layers 17,23 active (~25GB compile RAM)
                                 # 23 → layer 23 only  (~5GB compile RAM)

_all_titans_layers = (11, 17, 23)
active_titans_layers = tuple(l for l in _all_titans_layers if l >= titans_phase2_first_layer)
print(f"Active Titans layers: {active_titans_layers}")


In [ ]:
def load_titans_weights(load_dir: str):
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(os.path.abspath(load_dir))

titans_zip = "saved_titans_delta.zip"
titans_delta_path = "./saved_titans_delta"

if os.path.exists(titans_zip) and not os.path.exists(titans_delta_path):
    print(f"Unpacking {titans_zip}...")
    shutil.unpack_archive(titans_zip, titans_delta_path)

print("Loading Gemma base weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

print("Loading Phase 1 Titans weights...")
loaded_titans_params = load_titans_weights(titans_delta_path)

# Keep only weights for active Titans layers — inactive layers (< titans_phase2_first_layer)
# revert to original Gemma weights automatically via merge.
active_layer_keys = {f'layer_{l}' for l in active_titans_layers}
loaded_titans_params = {k: v for k, v in loaded_titans_params.items() if k in active_layer_keys}
print(f"Merging Titans weights for: {sorted(active_layer_keys)}")

merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)
print("✅ Phase 1 weights loaded and merged.")

## 3. Модель (training_phase=2)

In [ ]:

gemma_config = dataclasses.replace(
    Gemma3_1B_Titans.config,
    # sliding_window_size=128,
    training_phase=2,
    titans_layer_indices=active_titans_layers,
    titans_phase2_first_layer=titans_phase2_first_layer,
)

model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
)

## 4. Датасет

In [ ]:
import grain
import pickle

tokenizer = gm.text.Gemma3Tokenizer()
DATA_DIR = os.path.abspath('/content/drive/Shareddrives/shared veriga/trivia_qa_filtered')

def format_triviaqa(x):
    ctx = ""
    if x['search_results']['search_context']:
        ctx = x['search_results']['search_context'][0]
    q = x["question"]
    ans = x['answer']['value']
    x['src'] = f"Context: {ctx}\n\nQuestion: {q}"
    x['dst'] = ans
    return x

def get_precomputed_dataset(batch_size=8, tokenizer=None, max_length=2048, files=None):
    if files is None:
        files = ['test_gemma_generated.array_record']
    paths = [os.path.join(DATA_DIR, f) for f in files]
    print(f"Загрузка данных из: {paths}")

    class PickledArrayRecordDataSource(grain.python.ArrayRecordDataSource):
        def __getitem__(self, idx):
            return pickle.loads(super().__getitem__(idx))

    data_source = PickledArrayRecordDataSource(paths)

    return kd.data.py.DataSource(
        seed=42,
        data_source=data_source,
        shuffle=True,
        num_epochs=None,
        batch_size=batch_size,
        transforms=[
            format_triviaqa,
            gm.data.Seq2SeqTask(
                in_prompt="src",
                in_response="dst",
                out_input="tokens",
                out_target="target",
                out_target_mask="loss_mask",
                tokenizer=tokenizer,
                max_length=max_length,
                truncate=True,
            ),
            kd.data.py.Elements(keep=["tokens", "target"]),
        ],
    )

## 5. Оптимизатор (сниженный LR для дообучения)

In [ ]:
from adam_atan2 import adam_atan2
from optax.contrib._muon import MuonDimensionNumbers

def muon_only_dims(params):
    MUON_KEYS = {'to_queries', 'to_keys_values', 'combine_heads'}
    def _label(path, v):
        key    = str(path[-1].key) if hasattr(path[-1], 'key') else ''
        parent = str(path[-2].key) if len(path) > 1 and hasattr(path[-2], 'key') else ''
        if key == 'kernel' and parent in MUON_KEYS:
            return MuonDimensionNumbers(reduction_axis=0, output_axis=1)
        return None
    return jax.tree_util.tree_map_with_path(_label, params)

def is_muon_param(path_str):
    parts = path_str.split('/')
    return (len(parts) >= 2 and
            parts[-1] == 'kernel' and
            parts[-2] in {'to_queries', 'to_keys_values', 'combine_heads'})

def muon_mask(params):
    def _m(path, v):
        return is_muon_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

def adam_mask(params):
    def _m(path, v):
        return not is_muon_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

# Сниженные LR: веса уже предобучены в Phase 1
lr_muon = optax.warmup_cosine_decay_schedule(
    init_value=5e-4,
    peak_value=5e-4,
    warmup_steps=500,
    decay_steps=total_steps - 500,
    end_value=1e-5,
)

lr_adam = optax.warmup_cosine_decay_schedule(
    init_value=5e-4,
    peak_value=5e-4,
    warmup_steps=500,
    decay_steps=total_steps - 500,
    end_value=5e-5,
)
## lr for experiments 4e-4, 1.5e-4
inner_chain = optax.chain(
            optax.clip_by_global_norm(5.0),
            optax.masked(
            optax.contrib.muon(
                learning_rate=lr_muon,
                muon_weight_dimension_numbers=muon_only_dims,
                eps=1e-8,
                mu_dtype=jnp.float32,
        ),
        mask=muon_mask,
    ),
    optax.masked(
        adam_atan2(
            learning_rate=lr_adam,
            b1=0.80,
            b2=0.50,
            eps=1e-8,
            mu_dtype=jnp.float32,
        ),
        mask=adam_mask,
    ),
)

routing_optimizer = optax.MultiSteps(
    kd.optim.partial_updates(
        inner_chain,
        mask=kd.optim.select(["memory", "memory_gate"]),
    ),
    every_k_schedule=16,
)

## 6. Loss и метрики

In [ ]:
import flax
import flax.linen as nn
from kauldron import metrics as kd_metrics
from kauldron import kontext

class FullParamsInit(kd.ckpts.InitTransform):
    def __init__(self, params):
        self.params = params
    def transform(self, state: kd.train.TrainState) -> kd.train.TrainState:
        return state.replace(params=self.params)

# LM loss — основной сигнал Phase 2
train_losses = {
    "lm_loss": kd.losses.Value(
        values="preds.layer_losses['lm_loss']"
    )
}

# Метрики
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class MemoryGateMetric(kd_metrics.Metric):
    params: kd.kontext.Key = "params"

    @flax.struct.dataclass
    class State(kd_metrics.State):
        gate_metrics: flax.core.FrozenDict[str, jnp.ndarray] = flax.core.FrozenDict()
        def compute(self):
            return {k: np.array(v, dtype=np.float32) for k, v in self.gate_metrics.items()}
        @classmethod
        def empty(cls):
            return cls(gate_metrics=flax.core.FrozenDict())
        def merge(self, other):
            return self

    def get_state(self, params=None, **kwargs) -> State:
        if params is None:
            return self.State.empty()
        stats_dict = {}
        def find_gates(tree, path_prefix=""):
            if hasattr(tree, 'items'):
                for key, val in tree.items():
                    current_path = f"{path_prefix}_{key}" if path_prefix else str(key)
                    if key == "memory_gate":
                        mean_val = jnp.mean(val)
                        openness = jax.nn.sigmoid(mean_val)
                        stats_dict[f"titans_gates/{current_path}_raw_mean"] = mean_val
                        stats_dict[f"titans_gates/{current_path}_openness"] = openness
                    else:
                        find_gates(val, current_path)
        find_gates(params)
        return self.State(gate_metrics=flax.core.freeze(stats_dict))

@flax.struct.dataclass
class LRState(kd_metrics.State):
    lr_value: jnp.ndarray
    @classmethod
    def empty(cls):
        return cls(lr_value=jnp.array(0.0))
    def merge(self, other):
        return self
    def compute(self):
        return self.lr_value

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class LearningRateMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(lr_muon(step)))

train_metrics = {
    "memory_gate": MemoryGateMetric(),
    "learning_rate": LearningRateMetric(),
}

## 3. Модель (training_phase=2)

In [ ]:
# %cd /content/Titans_jax
# !git pull

In [ ]:
# import gemma_titans
# importlib.reload(gemma_titans)
# from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config

In [ ]:
# gemma_config = dataclasses.replace(
#     Gemma3_1B_Titans.config,
#     sliding_window_size=128,
#     training_phase=2,
#     titans_layer_indices=active_titans_layers,
#     titans_phase2_first_layer=titans_phase2_first_layer,
# )

# model = Gemma3_1B_Titans(
#     config=gemma_config,
#     dtype=jnp.bfloat16,
#     return_last_only=False,
#     tokens="batch.tokens",
# )

# print(f"Model: titans_layer_indices={gemma_config.titans_layer_indices}, phase2_first={gemma_config.titans_phase2_first_layer}")

## 7. Trainer

In [ ]:
workdir_name = f'titans_workdir_phase2_from{titans_phase2_first_layer}'

trainer = kd.train.Trainer(
    seed=42,
    workdir=os.path.abspath(f'./{workdir_name}'),
    train_ds=get_precomputed_dataset(
        batch_size=batch_size,
        tokenizer=tokenizer,
        max_length=max_length,
        files=['test_gemma_generated.array_record','train_gemma_generated.array_record'],#,'validation_gemma_generated.array_record'],
    ),
    model=model,
    init_transform=FullParamsInit(merged_params),
    num_train_steps=total_steps,
    train_losses=train_losses,
    train_metrics=train_metrics,
    optimizer=routing_optimizer,
    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),
)

print(f"Trainer initialized. workdir: {workdir_name}")

## 8. TensorBoard

Если у вас уже запущено длительное обучение trainer.train() в ячейке, и нужно запустить заново Tensorboard, вам нужно воспользоваться Терминалом Colab:

terminal command available for runtime restart:
```
fuser -k 6006/tcp && tensorboard --logdir /content/Titans_jax/titans_workdir_phase2_from11/ --port 6006 &
```
после чего обновить Tensorboard


In [ ]:
%reload_ext tensorboard
from tensorboard import notebook

# Показать список всех активных инстансов
notebook.list()



In [ ]:
!rm -rf /tmp/.tensorboard-info/*

In [ ]:
%tensorboard --logdir ./{workdir_name}/ --port=6006

## 9. Обучение

In [ ]:
state, aux = trainer.train()


## 10. Сохранение весов

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    _, titans_params = titans_tree_utils.split_titans_params(state.params)
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved Titans weights to {save_path}")
new_weights_name = f"saved_titans_phase2_from_{titans_phase2_first_layer}"

save_titans_weights(state, f"./{new_weights_name}")

shutil.make_archive(new_weights_name, 'zip', f"./{new_weights_name}")
print(f"\u2705 {new_weights_name}.zip ready")
destination_path = f"/content/drive/Shareddrives/shared veriga/saved_titans_delta/{new_weights_name}.zip"
shutil.copy(f"./{new_weights_name}.zip", destination_path)
print(f"Файл скопирован в: {destination_path}")